# LINT Pipeline

## Import

In [ ]:
from lint_ii import ReadabilityAnalysis
import pandas as pd
import numpy as np
import json
from enum import Enum
from typing import List

from anthropic import Anthropic
from openai import OpenAI
from google import genai
from google.genai import types
import lmstudio as lms
from pydantic import BaseModel, Field

from dotenv import load_dotenv
import os

## Rewrite Engines, Response schemas

### Engine

In [2]:
class RewriteEngine:
     """
     Engine that rewrites a text using a specific LLM
     
     
     """
     def __init__(self, 
                  model:str, 
                  name:str, 
                  is_dutch:bool = False, 
                  is_local:bool = False, 
                  is_open_source:bool = False, 
                  is_large:bool = False):
          
          self.model = model
          self.name = name
          self.is_dutch = is_dutch
          self.is_local = is_local
          self.is_open_source = is_open_source
          self.is_large = is_large
          pass
     
     def set_instruction(self, instruction:str):
          self.instruction = instruction
     
     def prompt_model(self, user_prompt:str):
          pass

In [162]:
# Response schema
# class RewriteChange(BaseModel):
#     original_part_of_text: str
#     rewritten_part_of_text: str
#     reason_for_change: str
class RewriteCategory(str, Enum):
    word_frequency = "verhoog woordfrequentie"
    syntactic_dependency_length = "verlaag afhankelijkheidslengte"
    content_words_per_clause = "verlaag contentwoorden per deelzin"
    proportion_concrete_nouns = "verhoog de verhouding van concrete zelfstandignaamwoorden"


class RewriteChange(BaseModel):
    change_id: str = Field("The id of the individual rewrite change.")
    start_word: int = Field("Inclusive start word index in orginal text")
    end_word: int = Field("Inclusive end word index in orginal text")
    original_text: str
    new_text: str
    reason_for_change: str = Field("Explanation for why the model made this change")
    rewrite_category: RewriteCategory = Field("Classification of the rewrite type.")

class RewriteSentence(BaseModel):
    original_sentence: str
    rewritten_sentences: str
    changes_in_sentence: List[RewriteChange] = Field("Every single change gets it's own RewriteChange object.")

class RewrittenText(BaseModel):
    text: List[RewriteSentence]
    text_genre: str

### LMStudio

In [163]:
# LMStudio hosts and manages all my local models. It optimizes gpu allocation so i don't have to.
class RE_LMStudio(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = lms.llm(self.model) # 'fietje-2-instruct'

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        chat = lms.Chat(self.instruction)
        chat.add_user_message(user_prompt)
        
        response = self.client.respond(
            history=chat,
            response_format=RewrittenText,
            config={
                "temperature": 1.0,
                # "maxTokens": 50,
            },
        )

        returnObject = None
        try:
            returnObject = RewrittenText.model_validate(response.parsed)
        except:
            print("Transforming to RewrittenText failed. Using default json schema instead.")
            returnObject = response.parsed

        return returnObject

### Claude

In [164]:
class RE_Claude(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        response = self.client.messages.parse(
            model=self.model, # "claude-opus-4-7"
            output_format=RewrittenText,
            system=self.instruction,
            max_tokens=1024,
            messages=[
                {
                    "role": "user",
                    "content": f"{user_prompt}",
                }
            ],
        )
        return response.parsed_output

### OpenAI

In [165]:
class RE_OpenAi(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)

    def prompt_model(self, user_prompt) -> str:
        response = self.client.responses.parse(
            model=self.model, # "gpt-4.1-nano"
            instructions=self.instruction,
            text_format = RewrittenText,
            input=[
                {
                    "role": "user",
                    "content": f"{user_prompt}",
                }
            ],
            reasoning={},
            tools=[],
            temperature=0.5,
            max_output_tokens=10000,
            top_p=1,
            store=True,
            include=["web_search_call.action.sources"]
            )
        return response.output_parsed

### Gemini

In [166]:
class RE_Gemini(RewriteEngine):


    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = genai.Client()

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        response = self.client.models.generate_content(
            model=self.model, # "gemini-3-flash-preview" 
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="low"),
                system_instruction=self.instruction,
                response_mime_type="application/json",
                response_schema=RewrittenText
            ),
            contents=f"{user_prompt}"
        )

        rewritten = RewrittenText.model_validate_json(response.text)
        return rewritten

## LINT Pipeline

In [215]:
class LintPipe:

    def __init__(self, instruction_en:str, instruction_nl:str, user_prompt:str):
        self.user_prompt = user_prompt
        self.analysis = self.get_lint_analysis(self.user_prompt)
        print(self.analysis)

        self.instruction_en = self.format_instruction(instruction_en)
        self.instruction_nl = self.format_instruction(instruction_nl, is_dutch = True)
        self.supra = None
        self.rewriteEngines = []
        self.prompt_responses = []
        return

    def format_instruction(self, instruction:str, is_dutch:bool = False):
        # Determine document and sentence scores:
        sent_scores = [int(sent_analysis['score']) for sent_analysis in self.analysis['sentence_stats'] if sent_analysis['score'] is not None]
        doc_score = str(int(self.analysis['document_stats']['document_lint_score']))
        
        if is_dutch:
            sent_scores = "{} en {}".format(", ".join(map(str, sent_scores[:-1])),  str(object=sent_scores[-1]))
        else:
            sent_scores = "{} and {}".format(", ".join(map(str, sent_scores[:-1])),  str(sent_scores[-1]))

        instruction = instruction.replace('<doc_score>', doc_score)
        instruction = instruction.replace('<sent_scores>', sent_scores)

        return instruction


    def add_engine(self, engine:RewriteEngine):
        # TODO: Retrieve all top_n uncommon 

        if engine.is_dutch:
            engine.set_instruction(self.instruction_nl)
        else:
            engine.set_instruction(self.instruction_en)
        self.rewriteEngines.append(engine)

    def list_engines(self):
        print("Rewrite engines:")
        for engine in self.rewriteEngines:
            print(f"NAME: {engine.name}, MODEL: '{engine.model}', ?NL {engine.is_dutch}, ?LOCAL: {engine.is_local}, ?OS {engine.is_open_source}, ?LARGE {engine.is_large}")

    def prompt_engines(self):
        # TODO: Prompt engines moet doen:
        # 1. Return een clean object:
        #     - Model id
        #     - Lint scores per zinsblok.
        #     - Herschreven tekst

        # - Dit object moet ook een methode hebben: extract_text en get_overall_lint_score()
        self.prompt_responses = []
        results = []
        if [] == self.rewriteEngines:
            print("No Rewrite Engines added. Use `add_engine` to do so.")
            return

        for engine in self.rewriteEngines:
            print(f"Rewriting using {engine.name} - Model: {engine.model}")
            engine_id = f'{engine.name}/{engine.model}'
            try:
                engine_result = engine.prompt_model(self.user_prompt)
                print(engine_result)
                self.supra = engine_result
                if type(engine_result) == RewrittenText:
                    print("is rewritten object")                    
                    engine_result = engine_result.model_dump()
                full_text = " ".join([rs['rewritten_sentences'] for rs in engine_result['text']])
                engine_result['full_text'] = full_text
                engine_result['engine'] = engine_id
                results.append(engine_result)
            except Exception as e:
                print(f"Something went wrong with {engine_id}, skipping...")
                print(e)
                faulty_engine = dict()
                faulty_engine['engine'] = engine_id
                faulty_engine['full_text'] = ''
                results.append(faulty_engine)
        print("Returning")
        self.prompt_responses = results.copy()
        return results

    def get_lint_analysis(self, user_prompt):
        analysis = ReadabilityAnalysis.from_text(user_prompt)
        return analysis.get_detailed_analysis()
    
    def eval_response(self):
        response_object = {}
        
        for resp in self.prompt_responses:
            print("bezig met", resp['engine'])
            analysis = ReadabilityAnalysis.from_text(resp['full_text'])
            doc_score = sent_scores = ""
            if analysis.lint.score == None:
                doc_score = sent_scores = np.nan
            else: 
                doc_score = str(int(analysis.lint.score))
                sent_scores = map(int, analysis.lint_scores_per_sentence)
                sent_scores = ", ".join(map(str, sent_scores))
            
            tr = {f"{resp['engine']} - text": resp['full_text'], f"{resp['engine']} - lint" : doc_score, f"{resp['engine']} - sent": sent_scores}
            print(tr)
            response_object.update(tr)

        return pd.DataFrame([response_object])

        


## Instructions

In [9]:
instruction_en = """
Goal: Your goal is to minimize the readability score of a text. The lower the score the better.

The readability score is calculated as follows: 100 - (
      - 4.21
      + (17.28 * word frequency)
      - (1.62  * syntactic dependency length)
      - (2.54  * content words per clause)
      + (16.00 * proportion concrete nouns)
  )

The current readability score of the text is <doc_score>, the goal is 34 or lower. The current scores per sentence are <sent_scores>.  

Method:
- First reason about what aspects of the text you would change, then perform the rewrite.
- Exclude short titles that aren't sentences.
- Do not rewrite sentences with a readability score lower than 34.
- Do not add information that wasn't there, leave the sentences as much in tact as possible to complete the task.
- Reduce words with low frequency use by finding synonyms or rewriting the sentence.
- Lower the syntactic dependency length by rewriting the sentence.
- Lower the content words per clause by splitting long sentences in smaller ones.
- Increase the proportion of concrete nouns by minimizing abstract nouns in the text.
- Perform this method per sentence.
- Format your response into the given structure.
- Give the reason for each specific change.


"""

In [ ]:
instruction_nl = """
Doel: Het doel is om de leesbaarheidsscore van een tekst zo laag mogelijk te maken. Hoe lager de score, hoe beter.
Gebruik de linguïstische metadata om moeilijke woorden en complexe zinsstructuren te herkennen.

De leesbaarheidsscore wordt als volgt berekend: 100 - (
        - 4.21
        + (17.28 * woordfrequentie)
        - (1.62 * syntactische afhankelijkheidslengte)
        - (2.54 * inhoudswoorden per bijzin)
        + (16.00 * aandeel concrete zelfstandige naamwoorden)
    )

De huidige leesbaarheidsscore van de tekst is <doc_score>; het doel is 34 of lager. De huidige scores per zin zijn <sent_scores>.

Methode:
- Denk eerst na over welke aspecten van de tekst je wil veranderen, voer daarna die veranderingen uit.
- Negeer korte titelteksten die geen zinnen zijn.
- Herschrijf geen zinnen met een leesbaarheidsscore lager dan 34.
- Voeg geen informatie toe die er niet stond; laat de zinnen zoveel mogelijk intact om de taak uit te voeren.
- Verminder het gebruik van woorden met een lage frequentie door synoniemen te gebruiken of de zin te herschrijven.
- Verlaag de syntactische afhankelijkheidslengte door de zin te herschrijven.
- Verlaag het aantal inhoudswoorden per bijzin door lange zinnen op te splitsen in kortere zinnen.
- Verhoog het aandeel concrete zelfstandige naamwoorden door abstracte zelfstandige naamwoorden zoveel mogelijk te vermijden.
- Pas deze methode per zin toe.
- Geef een reden voor elke specifieke aanpassing.
- Formatteer je antwoord in de gegeven structuur.
"""

In [14]:
inp_txt = """De Oudegracht is het sfeervolle hart van de stad.
In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. 
Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen."""

In [181]:
instruction_opt_nl = """
Doel: Het doel is om de leesbaarheidsscore van een tekst zo laag mogelijk te maken. Hoe lager de score, hoe beter.
Gebruik de linguïstische metadata om moeilijke woorden en complexe zinsstructuren te herkennen.

De leesbaarheidsscore wordt als volgt berekend: 100 - (
        - 4.21
        + (17.28 * woordfrequentie (mean_log_word_frequency, word_frequency))
        - (1.62 * syntactische afhankelijkheidslengte (max_sdl))
        - (2.54 * inhoudswoorden per bijzin (content_words_per_clause))
        + (16.00 * aandeel concrete zelfstandige naamwoorden (proportion_of_concrete_nouns))
    )

De huidige leesbaarheidsscore van de tekst is <doc_score>; het doel is 34 of lager. De huidige scores per zin zijn <sent_scores>.

Regels:
- Denk eerst na voor je de tekst herschrijft. Beredeneer waarom je een aanpassing maakt, verwijs naar de specifieke metadata.
- Bepaal eerst het thema van de tekst. Bepaal op basis daarvan of moeilijke woorden bepalend zijn voor de betekenis van de tekst.
- Herschrijf geen zinnen met een leesbaarheidsscore lager dan 34, behoud deze wel.
- Woorden met een lage woordfrequentie (dus word_frequency <= 3) zijn waarschijnlijk moeilijk. Vervang alleen deze door eenvoudigere alternatieven.
- Woorden met pos = ADJ mogen vereenvoudigd of verwijderd worden als ze niet essentieel zijn.
- Lange afhankelijkheden (dep_length groter dan 8) wijzen op complexe zinsstructuren, maak van de bijzin een aparte zin. Gebruik signaalwoorden tussen de nieuwe zinnen voor coherentie tussen de oude hoofdzin en de deelzinnen.
- Voorkom deelzinnen.
- Eigennamen (pos = PROPN) moeten behouden blijven.
- Houd de betekenis en de boodschap van de zinnen en de tekst hetzelfde.
- Voeg geen informatie toe.
- Geef de nieuwe tekst in het gegeven format.
"""

In [182]:
inp_text_evolved = inp_txt + """
METADATA:

[{'word_features': [{'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'oudegracht',
    'tag': 'N|eigen|ev|basis|zijd|stan',
    'pos': 'PROPN',
    'dep_length': 3,
    'super_sem_type': 'unknown'},
   {'text': 'is', 'tag': 'WW|pv|tgw|ev', 'pos': 'AUX', 'dep_length': 2},
   {'text': 'het', 'tag': 'LID|bep|stan|evon', 'pos': 'DET', 'dep_length': 1},
   {'text': 'sfeervolle', 'tag': 'ADJ|prenom|basis|met-e|stan', 'pos': 'ADJ'},
   {'text': 'hart',
    'tag': 'N|soort|ev|basis|onz|stan',
    'pos': 'NOUN',
    'word_frequency': 5.293120582960477,
    'super_sem_type': 'undefined'},
   {'text': 'van', 'tag': 'VZ|init', 'pos': 'ADP', 'dep_length': 1},
   {'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'stad',
    'tag': 'N|soort|ev|basis|zijd|stan',
    'pos': 'NOUN',
    'word_frequency': 5.435577664689725,
    'dep_length': 2,
    'super_sem_type': 'concrete',
    'punctuation': defaultdict(str, {'trailing': '.'})},
   {'text': '.', 'tag': 'LET', 'pos': 'PUNCT'}],
  'lint_score': 18.511612982419507,
  'difficulty_level': 1,
  'mean_log_word_frequency': 5.364349123825101,
  'max_sdl': 3,
  'proportion_of_concrete_nouns': 0.5,
  'content_words_per_clause': 4.0},
 {'word_features': [{'text': 'in',
    'tag': 'VZ|init',
    'pos': 'ADP',
    'dep_length': 1},
   {'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'middeleeuwen',
    'tag': 'N|soort|mv|basis',
    'pos': 'NOUN',
    'word_frequency': 3.423686536423184,
    'dep_length': 4,
    'super_sem_type': 'concrete'},
   {'text': 'was', 'tag': 'WW|pv|verl|ev', 'pos': 'AUX', 'dep_length': 3},
   {'text': 'het',
    'tag': 'VNW|pers|pron|stan|red|3|ev|onz',
    'pos': 'PRON',
    'dep_length': 2},
   {'text': 'hier',
    'tag': 'VNW|aanw|adv-pron|obl|vol|3o|getal',
    'pos': 'ADV',
    'dep_length': 1},
   {'text': 'een', 'tag': 'LID|onbep|stan|agr', 'pos': 'DET'},
   {'text': 'drukte',
    'tag': 'N|soort|ev|basis|zijd|stan',
    'pos': 'NOUN',
    'word_frequency': 3.96775458077346,
    'super_sem_type': 'abstract'},
   {'text': 'van', 'tag': 'VZ|init', 'pos': 'ADP'},
   {'text': 'belang',
    'tag': 'N|soort|ev|basis|onz|stan',
    'pos': 'NOUN',
    'word_frequency': 4.509063243912051,
    'dep_length': 1,
    'super_sem_type': 'abstract'},
   {'text': 'met', 'tag': 'VZ|init', 'pos': 'ADP', 'dep_length': 1},
   {'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'aan-', 'tag': 'SPEC|afgebr', 'pos': 'SYM', 'dep_length': 4},
   {'text': 'en', 'tag': 'VG|neven', 'pos': 'CCONJ'},
   {'text': 'afvoer',
    'tag': 'N|soort|ev|basis|zijd|stan',
    'pos': 'NOUN',
    'word_frequency': 3.3845344124610364,
    'dep_length': 6,
    'super_sem_type': 'concrete'},
   {'text': 'van', 'tag': 'VZ|init', 'pos': 'ADP'},
   {'text': 'goederen',
    'tag': 'N|soort|mv|basis',
    'pos': 'NOUN',
    'word_frequency': 3.7985612410265284,
    'dep_length': 3,
    'super_sem_type': 'undefined',
    'punctuation': defaultdict(str, {'trailing': '.'})},
   {'text': '.', 'tag': 'LET', 'pos': 'PUNCT'}],
  'lint_score': 54.27056340066443,
  'difficulty_level': 3,
  'mean_log_word_frequency': 3.816720002919252,
  'max_sdl': 6,
  'proportion_of_concrete_nouns': 0.4,
  'content_words_per_clause': 5.0},
 {'word_features': [{'text': 'nu', 'tag': 'BW', 'pos': 'ADV', 'dep_length': 4},
   {'text': 'is', 'tag': 'WW|pv|tgw|ev', 'pos': 'AUX', 'dep_length': 3},
   {'text': 'het',
    'tag': 'VNW|pers|pron|stan|red|3|ev|onz',
    'pos': 'PRON',
    'dep_length': 2},
   {'text': 'een', 'tag': 'LID|onbep|stan|agr', 'pos': 'DET', 'dep_length': 1},
   {'text': 'prachtige',
    'tag': 'ADJ|prenom|basis|met-e|stan',
    'pos': 'ADJ',
    'word_frequency': 4.7805033384066125},
   {'text': 'plek',
    'tag': 'N|soort|ev|basis|zijd|stan',
    'pos': 'NOUN',
    'word_frequency': 5.249034299064351,
    'super_sem_type': 'undefined'},
   {'text': 'om', 'tag': 'VZ|init', 'pos': 'ADP', 'dep_length': 1},
   {'text': 'te', 'tag': 'VZ|init', 'pos': 'ADP'},
   {'text': 'winkelen',
    'tag': 'WW|inf|vrij|zonder',
    'pos': 'VERB',
    'word_frequency': 4.177454440810221,
    'dep_length': 2},
   {'text': 'en', 'tag': 'VG|neven', 'pos': 'CCONJ', 'dep_length': 1},
   {'text': 'te', 'tag': 'VZ|init', 'pos': 'ADP'},
   {'text': 'lunchen',
    'tag': 'WW|inf|vrij|zonder',
    'pos': 'VERB',
    'dep_length': 5},
   {'text': 'of', 'tag': 'VG|neven', 'pos': 'CCONJ', 'dep_length': 1},
   {'text': 'te', 'tag': 'VZ|init', 'pos': 'ADP'},
   {'text': 'dineren',
    'tag': 'WW|inf|vrij|zonder',
    'pos': 'VERB',
    'word_frequency': 3.893254653252401,
    'dep_length': 8},
   {'text': 'in', 'tag': 'VZ|init', 'pos': 'ADP', 'dep_length': 2},
   {'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET', 'dep_length': 1},
   {'text': 'oude',
    'tag': 'ADJ|prenom|basis|met-e|stan',
    'pos': 'ADJ',
    'word_frequency': 5.436741798693928},
   {'text': 'stadskastelen',
    'tag': 'N|soort|mv|basis',
    'pos': 'NOUN',
    'word_frequency': 1.359228547196266,
    'dep_length': 3,
    'super_sem_type': 'unknown',
    'punctuation': defaultdict(str, {'trailing': '.'})},
   {'text': '.', 'tag': 'LET', 'pos': 'PUNCT'}],
  'lint_score': 63.24402181810589,
  'difficulty_level': 4,
  'mean_log_word_frequency': 4.149369512903963,
  'max_sdl': 8,
  'proportion_of_concrete_nouns': 0.0,
  'content_words_per_clause': 7.0}]


"""

In [15]:
T01_moei_txt = """Internationaal Monetair Fonds

Een orgaan dat nauw is betrokken bij de verbetering van de wereldhandelsstructuur, is het in 1994 opgerichte Internationaal Monetair Fonds (IMF). Net als de WTO organiseert het IMF regelmatig internationaal overleg en publiceert het regelmatig gegevens en voorspellingen over de wereldeconomie. De belangrijkste taak van het IMF is leningen verstrekken aan landen met een langdurig tekort op de betalingsbalans. De middelen daarvoor komen uit een fonds. (Een fonds is in dit geval een 'potje' met geld voor een bepaald doel). Daarin storten de lidstaten van het IMF geld.

Een langdurig tekort op de betalingsbalans ontstaat als een land gedurende lange tijd meer in- dan uitvoert. Het IMF biedt deze landen een lening aan. Aan de leningen zijn voorwaarden verbonden. Landen moeten het gebruiken om hun productiestructuur te verbeteren en zo toekomstige betalingsbalanstekorten zien te voorkomen. Ze moeten een plan voorleggen aan het IMF waarin staat hoe ze dat doel denken te bereiken. Als de productiestructuur inderdaad verbetert, kan het land op de wereldmarkt beter concurreren met andere landen. De export van het land kan stijgen en de import van het land afnemen. Het tekort op de betalingsbalans kan verminderen en uiteindelijk verdwijnen.

Net als de WTO draagt het IMF bij tot de instandhouding van de wereldhandel. Landen met een langdurig tekort op de betalingsbalans hebben te weinig officiële reserves om noodzakelijke goederen te importeren. Dankzij de leningen van het IMF kunnen ze geld blijven besteden bij andere landen. Landen als Rusland en de Oekraïne hebben de afgelopen jaren enorme bedragen bij het IMF geleend. Naast landen in Oost-Europa gebruiken vooral ontwikkelingslanden de kredieten van het IMF. De schuld van een aantal van deze landen aan het IMF is enorm hoog geworden. We kunnen spreken van een schuldencrisis. Regelmatig regelt het IMF dat hun schulden worden kwijtgescholden en/of verminderd."""

In [223]:
chatgpt = RE_OpenAi(
    model="gpt-4.1",
    name = "ChatGPT 4.1",
    is_local=False,
    is_open_source=False,
    is_large=True,
    is_dutch=True
)

chatgpt_51 = RE_OpenAi(
    model="gpt-5.1",
    name = "ChatGPT 5.1",
    is_local=False,
    is_open_source=False,
    is_large=True,
    is_dutch=True
)

gemini = RE_Gemini(
    model="gemini-3-flash-preview",
    name="Gemini",
    is_local=False,
    is_open_source=False,
    is_large=True,
    is_dutch=True
)

claude = RE_Claude(
    model='claude-sonnet-4-6',
    name="Claude",
    is_local=False,
    is_open_source=False,
    is_large=False,
    is_dutch=True
)

# fietje = RE_LMStudio(
#     model="fietje-2-instruct",
#     name="Fietje",
#     is_dutch=True,
#     is_local=True,
#     is_open_source=True,
#     is_large=False
# )

# mistral = RE_LMStudio(
#     model="mistralai/ministral-3-14b-reasoning",
#     name="Mistral",
#     is_dutch=True,
#     is_local=True,
#     is_open_source=True,
#     is_large=False
# )

geitje = RE_LMStudio(
    model="geitje-7b-ultra",
    name="Geitje",
    is_dutch=True,
    is_local=True,
    is_open_source=True,
    is_large=False
)

lc = lp.RE_Local(
    model="BramVanroy/fietje-2-instruct",
    name="Fietje",
    is_local=True,
    is_open_source=True,
    is_large=False
)

LMStudioWebsocketError: Client unexpectedly disconnected.

In [217]:
test_pipe = LintPipe("Herschrijf de tekst zodat die makkelijker te lezen is.", instruction_opt_nl , inp_txt)
# test_pipe.add_engine(chatgpt)
# test_pipe.add_engine(chatgpt_51)
# test_pipe.add_engine(gemini)
# test_pipe.add_engine(claude)
# test_pipe.add_engine(fietje)
# test_pipe.add_engine(geitje)
test_pipe.add_engine(lc)

{'document_stats': {'sentence_count': 3, 'document_lint_score': 48.20593518603563, 'document_difficulty_level': 3, 'min_lint_score': 18.511612982419507, 'max_lint_score': 63.24402181810589}, 'sentence_stats': [{'text': 'De Oudegracht is het sfeervolle hart van de stad.', 'score': 18.511612982419507, 'level': 1, 'mean_log_word_frequency': 5.364349123825101, 'top_n_least_freq_words': [('hart', 5.293120582960477), ('stad', 5.435577664689725)], 'proportion_concrete_nouns': 0.5, 'concrete_nouns': ['stad'], 'abstract_nouns': [], 'undefined_nouns': ['hart'], 'unknown_nouns': ['oudegracht'], 'max_sdl': 3, 'sdls': [{'token': 'de', 'dep_length': 0, 'heads': ['Oudegracht']}, {'token': 'oudegracht', 'dep_length': 3, 'heads': ['hart']}, {'token': 'is', 'dep_length': 2, 'heads': ['hart']}, {'token': 'het', 'dep_length': 1, 'heads': ['hart']}, {'token': 'sfeervolle', 'dep_length': 0, 'heads': ['hart']}, {'token': 'hart', 'dep_length': 0, 'heads': ['hart']}, {'token': 'van', 'dep_length': 1, 'heads': 

In [218]:
test_pipe.list_engines()

Rewrite engines:
NAME: Fietje, MODEL: 'BramVanroy/fietje-2-instruct', ?NL False, ?LOCAL: True, ?OS True, ?LARGE False


In [220]:
z = test_pipe.prompt_engines()

Rewriting using Fietje - Model: BramVanroy/fietje-2-instruct


/opt/homebrew/Caskroom/miniforge/base/envs/lint2test/lib/python3.11/site-packages/transformers/generation/utils.py:1590: UserWarning: Using the model-agnostic default `max_length` (=149) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


{ "text": [{ "original_sentence": "De Oudegracht is
Something went wrong with Fietje/BramVanroy/fietje-2-instruct, skipping...
string indices must be integers, not 'str'
Returning


In [219]:
z

[{'engine': 'Fietje/BramVanroy/fietje-2-instruct', 'full_text': ''}]

In [221]:
test_pipe.supra

'{ "text": [{ "original_sentence": "De Oudegracht is'

In [130]:
test_pipe.prompt_responses = z

In [187]:
test_pipe.eval_response()

bezig met ChatGPT 4.1/gpt-4.1
{'ChatGPT 4.1/gpt-4.1 - text': 'De Oudegracht is het hart van de stad. In de middeleeuwen was het hier druk. Er kwamen veel spullen aan en gingen weg. Nu is het een mooie plek. Je kunt er winkelen. Je kunt er eten in oude gebouwen.', 'ChatGPT 4.1/gpt-4.1 - lint': '17', 'ChatGPT 4.1/gpt-4.1 - sent': '14, 22, 11, 23, 11'}
bezig met ChatGPT 5.1/gpt-5.1
{'ChatGPT 5.1/gpt-5.1 - text': 'De Oudegracht is het sfeervolle hart van de stad. In de middeleeuwen was het hier erg druk met het brengen en halen van spullen. Nu is het een mooie plek. Je kunt er winkelen. Je kunt er lunchen. Je kunt er dineren in de oude kastelen.', 'ChatGPT 5.1/gpt-5.1 - lint': '22', 'ChatGPT 5.1/gpt-5.1 - sent': '18, 28, 23, 28'}
bezig met Gemini/gemini-3-flash-preview
{'Gemini/gemini-3-flash-preview - text': 'De Oudegracht is het sfeervolle hart van de stad. Vroeger was de gracht een drukke plek. Er kwamen veel spullen aan met boten. Nu is het een mooie plek om te winkelen. Je kunt er ook

,ChatGPT 4.1/gpt-4.1 - text,ChatGPT 4.1/gpt-4.1 - lint,ChatGPT 4.1/gpt-4.1 - sent,ChatGPT 5.1/gpt-5.1 - text,ChatGPT 5.1/gpt-5.1 - lint,ChatGPT 5.1/gpt-5.1 - sent,Gemini/gemini-3-flash-preview - text,Gemini/gemini-3-flash-preview - lint,Gemini/gemini-3-flash-preview - sent,Claude/claude-sonnet-4-6 - text,Claude/claude-sonnet-4-6 - lint,Claude/claude-sonnet-4-6 - sent
0,De Oudegracht is het hart van de stad. In de m...,17,"14, 22, 11, 23, 11",De Oudegracht is het sfeervolle hart van de st...,22,"18, 28, 23, 28",De Oudegracht is het sfeervolle hart van de st...,25,"18, 39, 20, 32, 12",De Oudegracht is het sfeervolle hart van de st...,17,"18, 7, 32, 22"


### Test

In [82]:
fietje = RE_LMStudio(
    model="fietje-2-instruct",
    name="Fietje",
    is_local=True,
    is_open_source=True,
    is_large=False
)
fietje.set_instruction(instruction_nl)

In [93]:
k = fietje.prompt_model(inp_txt)

In [91]:
k

RewrittenText(text=[RewriteResponse(original_sentence='De Oudegracht is het sfeervolle hart van de stad.', rewritten_sentences='Het is het sfeervolle hart van de stad.', changes_in_sentence=[]), RewriteResponse(original_sentence='In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', rewritten_sentences='Het was een drukke plek met de handel.', changes_in_sentence=[]), RewriteResponse(original_sentence='Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.', rewritten_sentences='Het biedt nu diverse mogelijkheden voor zowel winkelen als ontspannen.', changes_in_sentence=[])])

In [85]:
RewrittenText.model_validate(k)

RewrittenText(text=[RewriteResponse(original_sentence='In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', rewritten_sentences='[Hier worden alle zinnen herschreven om meer actueel en direct te zijn, bijvoorbeeld:] Tegenwoordig is de Oudegracht een populaire plek voor winkelen en culinaire ervaringen.', changes_in_sentence=[RewriteChange(original_part_of_text='De Oudegracht is het sfeervolle hart van de stad.', rewritten_part_of_text='De Oudegracht is het hart van de stad. Het is een plek waar mensen samenkomen om te ontspannen, winkelen en genieten van lokale delicatessen.', reason_for_change='Verbeter de zinsstructuur om een vloeiendere, beknopte formulering te creëren die meer overeenkomt met hedendaagse taalgebruik en gemakkelijker leesbaar is voor moderne lezers.')])])

In [207]:
inp_txt

'De Oudegracht is het sfeervolle hart van de stad.\nIn de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. \nNu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'

In [15]:
k = 'De Oudegracht is het sfeervolle hart van de stad.\nIn de middeleeuwen was het hier een drukte van belang met de <text_eval>aan- en afvoer<text_eval> van goederen. \nNu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'

In [81]:
# claude = RE_Claude(
#     model='claude-sonnet-4-6',
#     name="Claude",
#     is_local=False,
#     is_open_source=False,
#     is_large=False
# )
claude.set_instruction("Rewrite the text so that it is easier to understand. Only change text between <text_eval> tags.")
bf = claude.prompt_model(k)

In [82]:
bf

RewrittenText(text=[RewriteResponse(original_sentence='aan- en afvoer', rewritten_sentences='aanvoer en afvoer')])

In [98]:
llk = [1, 2, 3, 4]

for i in llk:
    if i < 4:
        continue
        print("dont print this")
    print("Element is groter dan of gelijk aan 4")

Element is groter dan of gelijk aan 4


In [21]:
analysis = test_pipe.analysis
with open("analysis.json", "w") as f:
    f.write(json.dumps(analysis, indent=4))


In [22]:
visual = ReadabilityAnalysis.from_text(inp_txt)
visual

ReadabilityAnalysis(['De Oudegracht is het sfeervolle hart van de stad.', 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', 'Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'])

In [23]:
print(visual)

ReadabilityAnalysis(['De Oudegracht is het sfeervolle hart van de stad.', 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', 'Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'])


In [79]:
visual_html = visual._repr_html_()

In [30]:
visual.as_dict()['sentences']

[{'word_features': [{'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'oudegracht',
    'tag': 'N|eigen|ev|basis|zijd|stan',
    'pos': 'PROPN',
    'dep_length': 3,
    'super_sem_type': 'unknown'},
   {'text': 'is', 'tag': 'WW|pv|tgw|ev', 'pos': 'AUX', 'dep_length': 2},
   {'text': 'het', 'tag': 'LID|bep|stan|evon', 'pos': 'DET', 'dep_length': 1},
   {'text': 'sfeervolle', 'tag': 'ADJ|prenom|basis|met-e|stan', 'pos': 'ADJ'},
   {'text': 'hart',
    'tag': 'N|soort|ev|basis|onz|stan',
    'pos': 'NOUN',
    'word_frequency': 5.293120582960477,
    'super_sem_type': 'undefined'},
   {'text': 'van', 'tag': 'VZ|init', 'pos': 'ADP', 'dep_length': 1},
   {'text': 'de', 'tag': 'LID|bep|stan|rest', 'pos': 'DET'},
   {'text': 'stad',
    'tag': 'N|soort|ev|basis|zijd|stan',
    'pos': 'NOUN',
    'word_frequency': 5.435577664689725,
    'dep_length': 2,
    'super_sem_type': 'concrete',
    'punctuation': defaultdict(str, {'trailing': '.'})},
   {'text': '.', 'tag': 'LET',

In [24]:
visual

ReadabilityAnalysis(['De Oudegracht is het sfeervolle hart van de stad.', 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', 'Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'])

In [198]:
from importlib import reload

In [202]:
import LintPipe as lp

In [204]:
reload(lp)

<module 'LintPipe' from '/Users/Michiel/Documents/Studie bachelor + master/Master ADS/Thesis/code/LintPipe.py'>

In [205]:
lc = lp.RE_Local(
    model="BramVanroy/fietje-2-instruct",
    name="Fietje",
    is_local=True,
    is_open_source=True,
    is_large=False
)

hic


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [192]:
test_pipe.list_engines()

Rewrite engines:
NAME: ChatGPT 4.1, MODEL: 'gpt-4.1', ?NL True, ?LOCAL: False, ?OS False, ?LARGE True
NAME: ChatGPT 5.1, MODEL: 'gpt-5.1', ?NL True, ?LOCAL: False, ?OS False, ?LARGE True
NAME: Gemini, MODEL: 'gemini-3-flash-preview', ?NL True, ?LOCAL: False, ?OS False, ?LARGE True
NAME: Claude, MODEL: 'claude-sonnet-4-6', ?NL True, ?LOCAL: False, ?OS False, ?LARGE False
